# Comprensión y recopilacion de datos de Sensores

## recopilacion de datos de Sensores

### Exploracion inicial de la arquitectura de Datos y Dependencias

Los registros históricos de sensores especificamente para combustible se almacenan en la tabla [jmineops].[rep].[custom_RepLibre_hist_Test_Combustible], generada mediante el procedimiento almacenado [rep].[sp_get_RepLibre_Test_Combustible]. Para identificar las fuentes primarias de datos, se ejecutó el siguiente script SQL que rastrea dependencias del objeto:

```sql
-- realizar una busqueda profunda del objeto, buscando donde minimamentefue refernciado el objeto custom_RepLibre_hist_Test_Combustible

SELECT 
	referencing_schema.name AS esquema,
	referencing_obj.name AS objeto,
	referencing_obj.type AS tipo_objeto
FROM
	sys.sql_modules AS mod
JOIN
	sys.objects AS referencing_obj ON mod.object_id = referencing_obj.object_id
JOIN
	sys.objects AS referencing_schema ON referencing_obj.schema_id = referencing_schema.schema_id
WHERE 
	mod.definition LIKE '%custom_RepLibre_hist_Test_Combustible%'
```

El sistema Hexagon Mining opera con sensores VIMS (Vital Information Management System) que centralizan datos operativos. Ademas de un subsistema de geolocalización opera bajo estándar GMS (Geospatial Mining System) esta ultima afirmacion sustentada en lo que se puede ver de lo que muestra el VNC de su sistema de geolocalizacion, ver la imagen de abajo, registrando coordenadas en latitud/longitud con precisión milimétrica (mm) y altura en centímetros (cm).

![imagen de sistema VNC](../imgs/WGS84.png)


**Tabla de Sensores:**  

| Sensor ID                    | Tipo          | Medición                          | Precisión                     | Observaciones Técnicas                                                                 |  
|------------------------------|---------------|-----------------------------------|-------------------------------|---------------------------------------------------------------------------------------|  
| `oem.vims. truck. fuel_gauge`   | Nivel Combustible | Porcentaje residual (pct)         | ±0.5%                        | Calcula el consumo mediante diferencial de porcentaje en intervalos de 15 segundos.    |    
| `oem.vims. truck. eng_spd`      | RPM Motor     | Revoluciones por minuto (rpm.5)   | Media revolución (±0.5 rpm)   | Error sistemático por resolución del sensor (ej: registro de 1800 rpm = 1800 ± 0.5).  |  
| `oem.vims. truck. ground_spd`   | Velocidad     | Kilómetros por hora (km/h)        | ±0.2 km/h                    | Validado contra sistema GPS integrado en flota.                                        |  

Los sensores clave identificados para la construcción del modelo predictivo son: el nivel de combustible, las revoluciones por minuto (RPM) y la velocidad del vehículo. Estos parámetros, junto con los datos del sistema de gestión (GMS), conforman la base técnica del modelo.

Se decidió excluir otros sensores, como el de carga útil, porque su información ya está representada de forma indirecta en otros conjuntos de datos que han sido previamente recolectados. En particular, los datos del modelo de ciclos de operación y del modelo de tiempos reflejan de manera precisa el uso y la carga de la maquinaria.

Incluir sensores adicionales no solo resultaría redundante, sino que también implicaría generar nuevamente datos que ya existen, han sido validados y cumplen con los estándares actuales. Esto supondría un esfuerzo innecesario al duplicar un trabajo ya realizado. Por esta razón, se considera que los datos obtenidos a partir de los ciclos de trabajo y del modelo de tiempos son confiables y no requieren una nueva validación.

### Exploracion de la base de datos

Si bien la información de los sensores más importantes para nuestro análisis (combustible, RPM y velocidad) se encuentra almacenada en la tabla principal llamada shift_sensors, para contextualizar estos datos es necesario cruzarla con otras tablas de la base de datos. Este proceso requiere un análisis detallado de las dependencias entre las diferentes tablas para identificar las relaciones esenciales que permiten interpretar correctamente cada lectura de sensor.

`A continuación se presentan las descripciones detalladas de cada tabla y su función dentro del sistema de monitoreo.`

Tabla: `enum_tables`

| Columna     | Descripción                                                                 | Ejemplo         | Relevancia para el Proyecto                                               |
|-------------|------------------------------------------------------------------------------|-----------------|---------------------------------------------------------------------------|
| `id`        | Identificador único autoincremental (clave primaria).                       | 1               | Clave primaria para identificar cada tipo enumerado                      |
| `updated_at`| Fecha y hora de la última actualización del registro.                       | 2022-11-17...   | Ayuda a auditar cambios y mantener trazabilidad                          |
| `type`      | Categoría del tipo enumerado (ej: `TopographyType`, `ProjectStatus`).       | TopographyType  | Agrupa los elementos de tipo enumerado para filtros o validaciones       |
| `name`      | Nombre legible del tipo de enumeración.                                     | Road            | Facilita visualización y comprensión de los registros                    |
| `symbol`    | Nombre clave del tipo de objeto del registro.                               | road            | Usado en lógica de programación o integraciones                          |
| `attributes`| Texto libre con atributos adicionales (no siempre usado).                   | -               | Puede contener metadatos complementarios                                 |
| `ordinal`   | Valor numérico para orden o jerarquía del elemento.                         | 1               | Permite ordenar visualmente o jerárquicamente los registros              |
| `deleted_at`| Marca de eliminación lógica (si aplica).                                    | NULL            | Útil para manejo de soft deletes                                         |
| `visual`    | Información visual o textual opcional del tipo enumerado.                   | NULL            | Complemento opcional, sin uso directo en análisis principal              |


Tabla: `equipment`

| Columna         | Descripción                                                                                  | Ejemplo  | Relevancia para el Proyecto                                                 |
|-----------------|-----------------------------------------------------------------------------------------------|----------|------------------------------------------------------------------------------|
| `id`            | Identificador único del equipo.                                                              | 286      | Clave primaria, base para análisis de uso y estado de equipos               |
| `type`          | Tipo de equipo.                                                                               | Truck    | Permite clasificar equipos (camión, pala, perforadora, etc.)               |
| `name`          | Nombre o código del equipo.                                                                   | T-210    | Identificador visual del equipo, común en reportes operacionales           |
| `created_at`    | Fecha de creación del registro.                                                               | 2021-11-10| Útil para trazabilidad y antigüedad del equipo                             |
| `updated_at`    | Fecha de última actualización.                                                                | 2025-02-28| Determina la vigencia del registro                                          |
| `deleted_at`    | Fecha de eliminación lógica (si aplica).                                                     | NULL     | Permite exclusión lógica sin pérdida de datos históricos                   |
| `revision`      | Número de revisión (puede estar vacío).                                                       | NULL     | Versionado del equipo o de su configuración                                |
| `device_id`     | ID del dispositivo asociado (relación externa).                                               | 567      | Asociación con sensores u otros dispositivos                               |
| `status_id`     | Estado actual del equipo (relación con enum).                                                 | 111      | Permite determinar si está activo, en mantenimiento, etc.                  |
| `reason_id`     | Razón del estado actual (ej: mantenimiento, en reserva, etc.).                               | 324      | Diagnóstico o causa asociada                                               |
| `lineup_status_id` | Estado en la secuencia operativa.                                                         | 113      | Parte del control de asignación de flota                                   |
| `equipment_type_id`| Clasificación del equipo, asociado a tabla `enum_tables`.                                 | 2772     | Categoriza técnicamente al equipo                                          |
| `activity_id`   | Actividad actual del equipo (clave externa).                                                 | 40       | Permite identificar su operación actual                                    |
| `activity_start`| Inicio de la actividad actual.                                                               | 2025-02-28| Útil para análisis de duración de actividades                              |
| `length`        | Longitud del equipo.                                                                         | 14       | Aporta en simulaciones físicas, distribución en el terreno                 |
| `unit_id`       | Unidad asignada al equipo (clave externa).                                                   | 15       | Asocia el equipo a una unidad funcional                                    |
| `project_id`    | Proyecto al que está asignado el equipo.                                                     | 185      | Permite agrupar datos por proyecto                                         |
| `prestart_check`| Indicador de verificación previa (1 = sí, 0 = no).                                           | 1        | Control de seguridad y cumplimiento de protocolos                          |
| `warnings`      | Advertencias activas (si las hay).                                                           | ""       | Puede alertar sobre condiciones anómalas                                   |

Tabla: `maintenance`

| Columna                     | Descripción                                                                 | Ejemplo            | Relevancia para el Proyecto                                          |
|-----------------------------|------------------------------------------------------------------------------|--------------------|----------------------------------------------------------------------|
| `id`                        | Identificador único de la orden de trabajo o registro de combustible.       | 285                | Clave primaria                                                       |
| `updated_at`                | Última actualización del registro.                                          | 2024-06-18         | Control de versiones y actualización de datos                       |
| `equipment_id`              | ID del equipo relacionado.                                                  | 309                | Relación directa con tabla `equipment`                              |
| `fuel_tank`                 | Capacidad del tanque de combustible (litros).                               | 4650               | Base para análisis de autonomía y consumo                           |
| `fuel_used`                 | Combustible consumido (litros).                                             | 0                  | Métrica clave de eficiencia                                         |
| `engine_hours`              | Horas acumuladas del motor (texto).                                         | 85597              | Usado para estimar mantenimiento y vida útil                        |
| `fuel_gauge`                | Porcentaje de medidor de combustible.                                       | 92                 | Indicador operacional útil para alertas                             |
| `engine_hours_updated_at`   | Fecha de última actualización de las horas del motor.                       | 2025-02-28         | Sincronización con operación real                                   |
| `fuel_gauge_updated_at`     | Última actualización del medidor de combustible.                            | 2025-02-28         | Verifica integridad de la información del sensor                    |
| `tkph_avg`, `tkph_max`      | Datos relacionados a carga por neumáticos (TKPH - ton-km/h).               | -                  | Útil para análisis de neumáticos                                    |
| `tire_time`, `tire_alarms`  | Información sobre neumáticos y sus alertas.                                | -                  | Apoya análisis de desgaste o problemas                              |
| `additives`, `comment`      | Campos de texto libre, sin descripción específica.                          | -                  | Podrían contener detalles adicionales de contexto operativo         |

Tabla: `sensor_sets`

| Columna             | Descripción                                                                 | Ejemplo                            | Relevancia para el Proyecto                                         |
|---------------------|------------------------------------------------------------------------------|------------------------------------|---------------------------------------------------------------------|
| `id`                | Identificador único del conjunto de sensores.                               | 2                                  | Clave primaria                                                      |
| `name`              | Nombre del conjunto de sensores.                                            | Snapshot: Brake Air Pressure       | Define la finalidad o categoría del set de sensores                |
| `updated_at`        | Fecha de actualización del set.                                             | 2021-12-09 21:32:03.000            | Permite validar vigencia de configuración                         |
| `deleted_at`        | Marca de eliminación lógica (si aplica).                                    | 2021-12-09 21:32:03.000            | Trazabilidad de versiones                                          |
| `device_id`         | ID del dispositivo que agrupa los sensores.                                 | 34                                 | Relación con un equipo físico o sensor gateway                     |
| `save_seconds`      | Intervalo de tiempo para guardar lecturas.                                  | -                                  | Define frecuencia de muestreo (no disponible en ejemplo)           |
| `sensors`           | IDs de sensores asociados, separados por coma.                              | 117,172,178                        | Define el conjunto real de sensores incluidos                      |
| `deadband_percent`  | Margen de cambio mínimo para considerar una lectura como válida.            | -                                  | Ayuda a evitar registros redundantes                               |
| `sensor_set_type_id`| Tipo de conjunto de sensores.                                               | 2247                               | Categorización técnica, permite agrupar tipos similares            |
| `expire_at`         | Fecha de expiración del set (si aplica).                                    | NULL                               | Define validez temporal de la configuración                        |

Tabla: `shift_sensors`

| Columna             | Descripción                                                                                             | Ejemplo              | Relevancia para el Proyecto                                               |
|---------------------|----------------------------------------------------------------------------------------------------------|----------------------|---------------------------------------------------------------------------|
| `id`                | Identificador único autoincremental del registro (clave primaria).                                       | 86233                | Clave principal para referenciar registros individuales de sensores      |
| `updated_at`        | Fecha y hora de la última actualización del registro en formato UTC-0.                                                    | 2022-10-27 14:55:59  | Trazabilidad de actualizaciones                                          |
| `created_at`        | Fecha y hora de creación del registro en formato UTC-0.                                                                   | 2022-10-27 14:55:55  | Útil para análisis de secuencia temporal                                 |
| `equipment_id`      | ID del equipo asociado (relacionado con la tabla `equipment`).                                           | 307                  | Vincula la medición a un equipo específico                               |
| `sensor_set_id`     | ID del conjunto de sensores utilizados.                                                                  | 473                  | Identifica qué grupo de sensores generó los datos                        |
| `sensor_alarm_id`   | ID de la alarma del sensor asociada (si aplica).                                                         | NULL                 | Permite relacionar eventos de alarma                                     |
| `all_values`        | Indica si se registraron todos los valores esperados del sensor (1 = Sí, 0 = No).                        | 1                    | Verifica integridad del conjunto de datos                                |
| `values`            | Valores crudos de sensores medidos (ej: "507,2265,509").                                                  | 507,2265,509,...     | Base para cálculos y análisis técnico                                    |
| `configs`           | Configuración aplicada a los sensores durante la medición (ej: códigos de parámetros).                   | 507,2265,...         | Permite reconstruir el contexto de la medición                           |
| `latitude`          | Latitud geográfica del equipo en formato entero (dividir entre 1,000,000).                              | -75948559            | Permite ubicar el equipo espacialmente                                   |
| `longitude`         | Longitud geográfica del equipo en formato entero (dividir entre 1,000,000).                             | -241961690           | Permite ubicar el equipo espacialmente                                   |
| `elevation`         | Elevación del equipo (en metros, ej: 406019 = 406.019 m).                                                | 406019               | Contexto geográfico para análisis de condiciones del terreno             |
| `active_alarms`     | Lista de alarmas activas separadas por coma.                                                             | "21706,18,585"       | Detecta condiciones críticas o anómalas durante el turno                 |
| `speed`             | Velocidad del equipo en la unidad definida (km/h, mph, etc.).                                            | 0                    | Indica comportamiento operativo                                           |
| `snapshot_id`       | ID del snapshot general de datos.                                                                        | 80331                | Permite vinculación con registros históricos                             |
| `shift_snapshot_id` | ID del snapshot específico del turno de trabajo.                                                         | NULL                 | Útil para análisis por turno                                             |
| `alarm_start_id`    | ID del evento que inició una alarma (si existe).                                                         | NULL                 | Permite trazar origen de condiciones anómalas                            |
| `engine_hours`      | Horas acumuladas de motor (ej: 80331 = 8033.1 horas si se divide por 10).                                | 80331                | Métrica de uso del equipo                                                |
| `sensor_health`     | Estado de salud del sensor (ej: porcentaje de integridad o código).                                      | NULL                 | Indica si el sensor funciona correctamente                               |
| `heading`           | Dirección de desplazamiento del equipo en grados (0–360°).                                               | NULL                 | Útil para análisis de rutas y patrones de movimiento                     |
| `odometer`          | Odómetro: distancia total recorrida por el equipo (km, millas, etc.).                                    | NULL                 | Mide desgaste y uso acumulado                                            |

Las tablas adicionales identificadas aportan información valiosa que contextualiza cada lectura de sensor. Por ejemplo, conocer el modelo del camión, su capacidad de tanque y otras características técnicas permite interpretar correctamente si una lectura de combustible indica un comportamiento normal o anómalo. Esta información complementaria es esencial para realizar un modelado preciso.

El análisis de las tablas de la base de datos revela patrones importantes sobre el funcionamiento de los tres sensores principales:
1. Nivel de Combustible (sensor_set_id = 481): Según los registros de la base de datos mas especificamente en la tabla `sensor_sets`, este sensor está configurado para registrar datos cada 30 segundos, proporcionando un monitoreo casi en tiempo real del estado del tanque.
2. Revoluciones del Motor o RPM (sensor_set_id = 490): Este sensor opera con una frecuencia menor, registrando información cada 60 segundos según la configuración establecida en la tabla sensor_sets.
3. Velocidad: Esta variable aparece en todos los registros de la tabla principal, lo que la convierte en el dato más consistente del conjunto.
   
Adicionalmente, todos los registros incluyen datos de ubicación geográfica (latitud, longitud y elevación), permitiendo conocer la posición exacta de cada vehículo en cada momento registrado.

Dado que los datos llegan con diferentes frecuencias (combustible cada 30 segundos, RPM cada 60 segundos), es necesario implementar una `Estrategia de Unificación de Datos` estrategia que busca combinarlos de manera coherente. La decisión se sustenta en varios factores:

- Mayor consistencia: Los datos de combustible presentan una regularidad superior
- Mejor disponibilidad: La revisión directa de los registros evidencia que los datos de RPM tienen vacíos significativos, llegando incluso a tener períodos de un mes completo sin información
- Frecuencia adecuada: Los intervalos de 30 segundos proporcionan una resolución temporal suficiente para los análisis requeridos

La estrategia implementada consiste en tomar cada registro de combustible y asociarlo con el dato de RPM más cercano en el tiempo hacia atrás. De esta manera, se obtienen registros completos cada 30 segundos que incluyen: nivel de combustible, RPM (el más reciente disponible), velocidad y coordenadas geográficas.

`Hallazgo critico`

Durante el análisis se descubrió una limitación fundamental: los datos de combustible y RPM no comenzaron a registrarse simultáneamente. Los registros de combustible están disponibles desde enero de 2024, mientras que los de RPM solo existen a partir de febrero de 2025.
Este hallazgo se determinó ejecutando consultas específicas para identificar las fechas mínimas de registro de cada tipo de sensor:

```sql
-- Búsqueda de datos mínimos de sensores RPM y combustible 
-- para determinar los rangos de tiempo disponibles

-- RPM
select min(created_at) from jmineops.dbo.shift_sensors
where created_at between '2024-01-01 11:00:00' and '2025-04-01 11:00:00'
and sensor_set_id = 490
-- Resultado: 2025-02-04 21:44:00.000

-- Combustible  
select min(created_at) from jmineops.dbo.shift_sensors
where created_at between '2024-01-01 11:00:00' and '2025-04-01 11:00:00'
and sensor_set_id = 481
-- Resultado: 2024-01-27 19:51:00.000
```

Esta disparidad temporal define claramente los períodos de tiempo que pueden analizarse con datos completos de ambos sensores.

`Ajuste de Zona Horaria`

Los datos se almacenan en la base de datos usando el tiempo UTC+0 (tiempo universal coordinado), pero requieren conversión a UTC-4 (horario boliviano) para alinearse con los turnos operativos reales de la empresa. Esta conversión es fundamental porque los turnos de trabajo se definen según el horario local:

Turno diurno: de 07:00 a 18:59 horas
Turno nocturno: de 19:00 a 06:59 horas

Sin esta corrección, la clasificación de actividades según el turno resultaría incorrecta, afectando la precisión de los análisis posteriores.

Como ultimo aspecto que se toma en cuenta es la oportumidad de clasificar datos al obtenerlos, lo que pasa por distinguir el estados de ralenti en los vehículos, debido a que tenemos los datos necesarios para calcularlo. Para la diferenciación entre Ralentí y Motor Apagado: Un vehículo se considera en "ralentí" cuando el motor está funcionando pero sin movimiento (durante cargas, esperas, etc.). Se diferencia del estado "apagado" y en "movimiento" mediante criterios que se detallaran mas adelante.

Esta lógica se basa en un procedimiento ya validado existente en la base de datos llamado [mscjmina].[dbo].[msc_KPI_Ralenti], que había sido desarrollado y probado previamente por el equipo técnico de la empresa.

### Esquema Final de Relaciones

Con todos estos elementos analizados y procesados, se obtuvo un esquema coherente que muestra cómo se relacionan las tres variables críticas (combustible, RPM y velocidad) con toda la información contextual necesaria.

El diagrama final desarrollado ilustra estas relaciones de manera visual, facilitando la comprensión del flujo de datos desde los sensores hasta el análisis final:

![imagen de diagrama entidad relacion de sensores](../imgs/ER_sensors.png)

### Obtención del Dataset

El procedimiento almacenado desarrollado `[Model].[Sensor]` para extraer los datos maneja dos escenarios temporales distintos basados en la disponibilidad de información de los sensores. Para fechas anteriores a febrero de 2025, cuando solo están disponibles los datos de combustible, el sistema se enfoca exclusivamente en este sensor, calculando el consumo en litros utilizando la capacidad del tanque documentada en las tablas de mantenimiento y clasificando el estado operativo entre ralentí o movimiento según la velocidad registrada.

Para fechas posteriores a febrero de 2025, cuando ya se cuenta con datos de ambos sensores, el sistema integra dinámicamente la información de RPM mediante una tabla temporal que correlaciona las lecturas de combustible y RPM. Esta correlación se realiza buscando para cada registro de combustible el valor de RPM más reciente disponible, asegurando que cada registro final contenga información completa y coherente.

La clasificación de estados operativos se basa en criterios técnicos específicos que distinguen entre diferentes situaciones del vehículo:
- **Movimiento**: Cuando se registra velocidad mayor a cero
- **Ralentí**: Motor funcionando sin movimiento (intervalos menores a cinco minutos sin actividad)
- **Apagado**: Intervalos superiores a cinco minutos sin actividad combinados con RPM menores a 300

Los datasets resultantes se estructuran con una nomenclatura estandarizada que incluye el identificador del equipo, el tipo de sensor y el período analizado (por ejemplo, `T-243_fuel_2024-01`). Esta estandarización facilita la trazabilidad y el procesamiento posterior por herramientas de análisis predictivo.

El procedimiento se ejecuta mediante parámetros específicos como fecha de inicio, fecha de fin y identificador del equipo, generando salidas estructuradas. A continuación se muestra la consulta sql creada:

```sql
USE Msc_Dev
GO
CREATE OR ALTER PROCEDURE [Model].[Sensor]
    @FechaIni DATETIME,
    @FechaFin DATETIME,
    @EquipmentID INT
AS
BEGIN
    SET NOCOUNT ON;

    -- Ajustar a UTC+0 (los datos están en UTC)
    SET @FechaIni = DATEADD(HOUR, 4, @FechaIni);
    SET @FechaFin = DATEADD(HOUR, 4, @FechaFin);

    DECLARE @FuelTank DECIMAL(10,2);
    DECLARE @Model VARCHAR(15);
    DECLARE @EquipmentName VARCHAR(100);

    -- Obtener capacidad del tanque de combustible
    SELECT @FuelTank = fuel_tank 
    FROM [jmineops].[dbo].[maintenance] WITH (NOLOCK)
    WHERE equipment_id = @EquipmentID;

    -- Obtener modelo y nombre del equipo
    SELECT 
        @Model = et.name,
        @EquipmentName = e.name
    FROM [jmineops].[dbo].[equipment] e WITH (NOLOCK)
    LEFT JOIN [jmineops].[dbo].[enum_tables] et WITH (NOLOCK)
        ON e.equipment_type_id = et.id
    WHERE e.id = @EquipmentID
    AND et.name IN ('CAT 789C', 'CAT 793D');

    -- Seleccionar según la fecha
	-- busca los datos minimos de sensores rpm y fuel para determinar los rangos de tiempo desde el que se tomaran en cuenta en los datasets
	-- rpm, por esta razon es la condicion if
	-- select min(created_at) from jmineops.dbo.shift_sensors
	-- where created_at between '2024-01-01 11:00:00' and '2025-04-01 11:00:00'
	-- and sensor_set_id = 490
	-- resultado
	-- 2025-02-04 21:44:00.000
    IF @FechaFin < '2025-02-04'
    BEGIN
        -- SOLO SENSOR DE COMBUSTIBLE
        SELECT
            ShiftDate = CONVERT(DATE, DATEADD(HOUR, -4, f.created_at)),
            Shift = CASE
                        WHEN CAST(DATEADD(HOUR, -4, f.created_at) AS TIME) 
                            BETWEEN '07:00:00' AND '18:59:59' THEN 'D'
                        ELSE 'N'
                    END,
            Timestamp = DATEADD(HOUR, -4, f.created_at),
            RecordDuration = DATEDIFF(SECOND, LAG(f.created_at) OVER(ORDER BY f.created_at), f.created_at),
            Equipment = @EquipmentName,
            TruckFleet = ISNULL(@Model, 'NULL'),
            FuelLevel = ISNULL(f.[values], 0),
            FuelLevelLiters = ISNULL((@FuelTank * TRY_CAST(f.[values] AS DECIMAL(10,2)) / 100), 0),
            FuelGauge = CASE 
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 10 THEN 'Critical'
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 30 THEN 'Low'
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 50 THEN 'Medium'
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 80 THEN 'High'
                          WHEN f.[values] IS NULL THEN 'NoData'
                          ELSE 'Full'
                        END,
            Speed = ISNULL(f.speed, 0),
            RPM = 0,
            Ralenti = CASE 
                WHEN f.speed = 0 THEN 'Ralenti'
                WHEN f.speed > 0 THEN 'Moviendose'
                ELSE 'Otro'
            END,
            Latitude = f.latitude,
            Longitude = f.longitude,
            Elevation = f.elevation
        FROM jmineops.dbo.shift_sensors f WITH (NOLOCK)
        WHERE f.sensor_set_id = 481
          AND f.created_at BETWEEN @FechaIni AND @FechaFin
          AND f.equipment_id = @EquipmentID
        ORDER BY f.created_at;
    END
    ELSE
    BEGIN
	    -- Crear tabla temporal en lugar de variable de tabla
		DROP TABLE IF EXISTS #TempRPMData
		-- Tabla para almacenar resultados temporales
		CREATE TABLE #TempRPMData (
			created_at DATETIME,
			equipment_id INT,
			[values] DECIMAL(10,2),
			speed DECIMAL(10,2)
		);

	    -- Obtener datos de RPM usando el procedimiento más eficiente
	    INSERT INTO #TempRPMData
        EXEC [Model].[explore_data]
            @Eq = @EquipmentID,
            @FechaIni = @FechaIni,
            @FechaFin = @FechaFin,
            @SensorSetId = 490;
        -- SENSOR DE COMBUSTIBLE + RPM
        WITH SensorFuel AS (
            SELECT 
                created_at,
                equipment_id,
                [values],
                speed,
                latitude,
                longitude,
                elevation
            FROM jmineops.dbo.shift_sensors WITH (NOLOCK)
            WHERE sensor_set_id = 481
              AND created_at BETWEEN @FechaIni AND @FechaFin
              AND equipment_id = @EquipmentID
        )
        SELECT
            ShiftDate = CONVERT(DATE, DATEADD(HOUR, -4, f.created_at)),
            Shift = CASE
                        WHEN CAST(DATEADD(HOUR, -4, f.created_at) AS TIME) 
                            BETWEEN '07:00:00' AND '18:59:59' THEN 'D'
                        ELSE 'N'
                    END,
            Timestamp = DATEADD(HOUR, -4, f.created_at),
            RecordDuration = DATEDIFF(SECOND, LAG(f.created_at) OVER(ORDER BY f.created_at), f.created_at),
            Equipment = @EquipmentName,
            TruckFleet = ISNULL(@Model, 'NULL'),
            FuelLevel = ISNULL(f.[values], 0),
            FuelLevelLiters = ISNULL((@FuelTank * TRY_CAST(f.[values] AS DECIMAL(10,2)) / 100), 0),
            FuelGauge = CASE 
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 10 THEN 'Critical'
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 30 THEN 'Low'
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 50 THEN 'Medium'
                          WHEN TRY_CAST(f.[values] AS DECIMAL(5,2)) <= 80 THEN 'High'
                          WHEN f.[values] IS NULL THEN 'NoData'
                          ELSE 'Full'
                        END,
            Speed = COALESCE(NULLIF(r.speed, 0), f.speed, 0),
            RPM = ISNULL(TRY_CAST(r.[values] AS DECIMAL(10, 2)), 0),
            Ralenti = CASE 
                WHEN f.speed = 0 AND DATEDIFF(MINUTE, LAG(f.created_at) OVER (ORDER BY f.created_at), f.created_at) BETWEEN 0 AND 5 THEN 'Ralenti'
                WHEN f.speed = 0 AND DATEDIFF(MINUTE, LAG(f.created_at) OVER (ORDER BY f.created_at), f.created_at) > 5 THEN 'Apagado'
                WHEN f.speed > 0 THEN 'Moviendose'
                WHEN f.speed IS NULL AND DATEDIFF(MINUTE, LAG(f.created_at) OVER (ORDER BY f.created_at), f.created_at) > 5 THEN 'Apagado'
                WHEN f.speed IS NULL AND DATEDIFF(MINUTE, LAG(f.created_at) OVER (ORDER BY f.created_at), f.created_at) BETWEEN 0 AND 5 AND r.[values] >= 300.0 THEN 'Ralenti'
                WHEN f.speed IS NULL AND DATEDIFF(MINUTE, LAG(f.created_at) OVER (ORDER BY f.created_at), f.created_at) BETWEEN 0 AND 5 AND r.[values] < 300.0 THEN 'Apagado'
                ELSE 'Otro'
            END,
            Latitude = f.latitude,
            Longitude = f.longitude,
            Elevation = f.elevation
        FROM SensorFuel f
        OUTER APPLY (
            SELECT TOP 1 
                [values], 
                speed 
            FROM #TempRPMData  r
            WHERE r.created_at <= f.created_at
            ORDER BY r.created_at DESC
        ) r
        ORDER BY f.created_at;
	    -- Eliminar la tabla temporal al finalizar
        DROP TABLE #TempRPMData;
    END
END;
GO

select * from jmineops.dbo.equipment

EXEC [Model].[Sensor] 
    @FechaIni = '2025-08-01 07:00:00',
    @FechaFin = '2025-09-01 06:59:59',
    @EquipmentID = 311

select * from jmineops.dbo.shift_sensors
where equipment_id = 290 
and sensor_set_id = 490
and created_at between '2025-02-01 08:00:00' and '2025-02-04 20:00:00'
```


## Descripción del Dataset

**Selección de camiones para el análisis:**  
Para este estudio, elegimos los datos de los camiones T-210 a T-215 porque es nuestra muestra escogida. El enfoque se centra específicamente en el **nivel de combustible** (medido en litros), ya que es el dato central para nuestro análisis. Otros datos como las RPM o el modelo del camión son contextualmente importantes, pero secundarios para este caso.  

### Series temporales del nivel de combustible

**Patrón típico:**  
El nivel de combustible normalmente **disminuye gradualmente** porque el camión consume combustible durante sus operaciones. Sin embargo, el sensor de nivel (no de litros exactos) registra fluctuaciones de **±10 a ±20 litros**. Esto ocurre porque el movimiento del camión hace que el combustible se desplace dentro del tanque, afectando las lecturas del sensor.  

> *En la siguiente gráfica del camion T-210 en fecha 2 de febrero de 2024 se observa:*
> - *Tendencia general de descenso*  
> - *Pequeñas subidas/bajadas por movimiento del camión*  
> - *Recargas cuando el nivel esta cerca de los 1,000 litros*  

![imagen de sistema VNC](../imgs/t210_2_2_2024.png)

### Análisis de cambios en el combustible (delta)  

El denominado **"delta de combustible"** en nuestro contexto es el **cambio entre mediciones consecutivas** del nivel de combustible. Por ejemplo: si a las 10:00 hay 1,500 L y a las 10:30 hay 1,480 L, el delta es de -20 L. Analizando esta nueva variable calculada se obtiene los siguietes **hallazgos clave**: 

El histograma de frecuencias del camión T-210 como muestra, al analizar todos los registros del mismo revela que:  
- **85,709 registros** muestran aumentos de +10 a +20 L  
- **102,785 registros** muestran disminuciones de -20 a -10 L  

![histograma de frecuencias de delta](../imgs/t210_deltafuel_hist.png)

Se uso el método `sqrt` (raíz cuadrada del total de datos) para definir los intervalos del histograma. Esto generó **665 segmentos**, ofreciendo el mejor equilibrio entre detalle y claridad entre los métodos evaluados. Entre otros métodos evaluados (*auto, fd, scott, sturges*), el `sqrt` demostró mayor efectividad.

La implementación técnica incluye funciones como calculo de bins dinamico, para mas detalles respecto a la implementacion se presenta enlace al codigo escrito en python almacenado en el siguiente repositorio de github [`sensor_data_eda`](https://github.com/JoseIgnacioFernandezMamani/FuelOptiMine/blob/c0e1fa937af709a427e180cf5de4046c8b012b85/frontend/web/app/analitycs/EDA/sensor/sensor_data_eda.py#L65).

Además de las fluctuaciones normales, se detectaron anomalías críticas que deberan detallarse cada una para comprender la naturaleza de los datos


## Tipos de anomalías detectadas del sensor de combustible

### Caidas repentinas del sensor o barrancos

Son fallas momentáneas del sensor donde el combustible muestra **0 litros brevemente** o muestra una caida injustificada que sale de la tendencia normal y luego recupera los valores de la tendencia normal.
   
   - *Ejemplo:* La muestra es del Camión T-210 (02/03/2025, 16:30).  
   - *Patrón:* El patrón muestra tres puntos: `valor en la tendencia normal` → `caída abrupta` → `recuperación a la tendencia normal`.

![barranco](../imgs/t210_3_2_2024.png)

### Subidas repentinas del sensor o picos

Son fallas momentáneas del sensor donde el combustible muestra **Lecturas anómalamente altas** injustificadas que salen de la tendencia normal y luego recupera los valores de la tendencia normal. 

   - *Ejemplo:* Camión T-213 (04/04/2024, 05:13 AM).
   - *Patrón:* El patrón muestra tres puntos: `valor en la tendencia normal` → `salto ilógico` → `recuperación a la tendencia normal`. Es en si el opuesto a los barrancos.
       
![pico](../imgs/t213_4_3_2024.png)

### Ruido del sensor

Son oscilaciones erráticas (subidas/bajadas rápidas) causadas por interferencias, .  

   - *Ejemplo:* Camión T-214 (16/02/2024).  
   - *Patrón:* El patrón muestra cuatro puntos que forman una "Z" en el caso de ser el mas pequeño ruido, en el caso de ser un ruido grande el patrón muestra una forma del simbolo de  resistencia eléctrica. A considerar es que el patron de Z es suficiente para que se detecten desde los mas pequeños ruidos a los mas grandes.
     
![ruido](../imgs/t214_16_2_2024.png)

### Caidas seguidas de una tendencia baja o valles

Anomalías donde el nivel de combustible cae abruptamente y se mantiene bajo durante varios registros (mínimo 2 mediciones consecutivas), sin recuperarse inmediatamente.

    - *Ejemplo:* Camión T-212 (08/03/2024).
    - *Patrón:* El patrón muestra cuatro puntos que forman una "U" siendo 2 puntos en la tendencia normal y minimo 2 o varios en la tendencia baja del valle.

### Subidas seguidas de una tendencia alta o mesetas

Incrementos anómalos donde el combustible sube y se mantiene alto de forma persistente (mínimo 2 registros), sin justificación operativa.

    - *Ejemplo:* Camión T-212 (08/03/2024).
    - *Patrón:* El patrón muestra cuatro puntos que forman una "n" siendo 2 puntos en la tendencia normal y minimo 2 o varios en la tendencia alta de la meseta.

Ambos anomalias afortunadamente no son muy comunes, pero se presentaron de forma extraordinaria en la misma siguiente muestra:

![ruido](../imgs/t212_8_3_2024.png)

## Eventos de recarga de combustible

Ahora una vez mencionado las anomalias es mas que importante diferenciar que en cuanto a eventos de recarga hay 2 diferencias, puede parecer que solo debe ser uno, pero estan 2 tipos de comportamiento en un evento de recarga de combustible, son los siguientes:

### Recarga rápida (común)  

Este patrón común ocurre en solo 2-3 registros en un tiempo de entre 5 a 8 minutos. Presenta dos escenarios claros: 
       
        Escenario A:
        Punto 1: Nivel normal (ej: 1,200 L) → Punto 2: Salto brusco (ej: 2,950 L)
        Causa: El sensor no registra el instante en el que el motor fue apagado.

        Escenario B:
        Punto 1: Nivel normal (ej: 1,150 L) → Punto 2: 0 L (apagado) → Punto 3: Nivel lleno (ej: 3,200 L)
        Causa: El sensor registra el instante en el que el motor fue apagado por lo tanto obtiene 0 en sus registros.
        
En ambos casos se cumple el procedimiento estándar según manual técnico de, apagar el motor antes y durante la recarga de combustible.

    - *Ejemplo:* Camión T-212 (08/03/2024).

![recarga rapida de combustible](../imgs/t214_22and27_2_2024.png)

### Recarga constante:

Patrón inusual ocure en 2 a mas registros en un periodo de 5 a 8 minutos, donde el combustible aumenta gradualmente en incrementos superiores a 190 litros del de nuestro umbral, aunque en ocasiones se registra incrementos inferiores a nuestro umbral. Una posible causa puede ser la recarga con motor encendido o un evento que cause que el sensor continue operando,. La imagen requiere ampliación para visualizar este patrón sutil a simple vista.

![recarga rapida de combustible](../imgs/t214_16_10_2024.png)



Antes de determinar cómo se obtiene la lógica detrás de la detección de eventos de recarga y normalizar los registros, es necesario obtener un último punto más, que hace referencia a los aspectos técnicos de este mismo, específicamente la capacidad de combustible de los camiones y cuál es el umbral desde el que se considera una recarga en sí.

## Detalles tecnicos

`capacidad de los tanques de combustible`

Los camiones analizados muestran tres configuraciones distintas: con capacidades (3200L en T-210 a T-215) siendo el camion T-213 la excepcion con 3790 en T-213 y de 4354L en T-234 a T-243. Estas variaciones son cruciales para establecer valores de referencia durante el análisis de recargas y detección de anomalías.

| Truck  | Capacity (liters) | Engine Hours |
|--------|-------------------|--------------|
| T-233  | 3200              | 80780.05     |
| T-232  | 3200              | 81615.79     |
| T-231  | 3200              | 91142.87     |
| T-230  | 3200              | 85705.91     |
| T-225  | 3200              | 62122.16     |
| T-224  | 3200              | 86498.91     |
| T-223  | 3200              | 83038.48     |
| T-222  | 3200              | 101721.6     |
| T-221  | 3200              | 95410.52     |
| T-220  | 3200              | 98834.77     |
| T-219  | 3200              | 98987.12     |
| T-218  | 3200              | 101089.7     |
| T-217  | 3200              | 96873.76     |
| T-216  | 3200              | 95502.92     |
| T-215  | 3200              | 81657.15     |
| T-214  | 3200              | 88599.7      |
| T-213  | 3790              | 100514       |
| T-212  | 3200              | 92162.23     |
| T-211  | 3200              | 86432.77     |
| T-210  | 3200              | 87159.49     |
| T-234  | 4354              | 63759.72     |
| T-235  | 4354              | 64259.77     |
| T-236  | 4354              | 68060.97     |
| T-237  | 4354              | 69173.2      |
| T-238  | 4354              | 70327.46     |
| T-239  | 4354              | 64386.07     |
| T-240  | 4354              | 65289.88     |
| T-241  | 4354              | 70422.64     |
| T-242  | 4354              | 68354.99     |
| T-243  | 4354              | 63857.04     |

`umbral de recarga de combustible`


Para identificar eventos genuinos de recarga de combustible, establecimos un umbral mínimo basado en las especificaciones técnicas de los surtidores. Según el manual de operaciones `3.04.P36.I32_Abastecimiento_Combustible_Equipos_Mina_Rev.9.pdf`, se presentan tres patrones de caudal:  

**CARACTERÍSTICAS TÉCNICAS DE ABASTECIMIENTO DE COMBUSTIBLE**

- **Caudal nominal general**: 114-568 L/min (promedio: 341 L/min)  
- **Aforo STT**: 388 L/min  
- **Aforo Kenworth**: 368 L/min
  
Entonces para calcular un umbral de acorde se realiza los siguiente:

1. Calculamos un caudal promedio integrado de:  
(341 + 388 + 368) / 3 = **365.66 L/min**  

2. Convertido a intervalos de 30 segundos (periodo entre mediciones):  
365.66 L/min ÷ 2 = **182.83 L/30s**  

**Decisión técnica:**  
Establecimos un umbral conservador de **190 L/30s** (redondeo superior a la decena) fundamentada porque permite diferenciar claramente fluctuaciones normales de (±10-20 L).

## Detección de eventos de recarga

Primeramente mencionar que para mayor detalle de la logica exacta, se entrega el repositorio donde se encuentra la implementacion del codigo: [repositorio](https://github.com/JoseIgnacioFernandezMamani/FuelOptiMine.git) La lógica para identificar recargas válidas se basa en lo siguiente:

**1. Preparación de datos:**
- Se cargan los datos a nuestro dataset de python.
- Se calcula el cambio de combustible entre mediciones (`DeltaFuel`)
- Se detectan apagados del sensor (cuando el combustible cae a 0 abruptamente) o sube a un valor imposible mayor a la capacidad del tanque.
- Se ordenan todos los registros por fecha y hora

**2. Eliminación de anomalías:**
El sistema identifica y elimina falsos positivos usando estas reglas:
- **Picos/barrancos:** Cambios bruscos usando el umbral (+/-190L).
- **Ruido:** Oscilaciones rápidas (sube-baja-sube o baja-sube-baja), las oscilaciones tambien se determinan usando el umbral.
- **Mesetas/valles:** Niveles artificialmente altos o bajos que persisten, tomando como referencia la [mediana móvil](https://es.wikipedia.org/wiki/Media_m%C3%B3vil) de valores anteriores y posteriores

**3. Detección de recargas:**
Se aplican dos métodos complementarios:

**a) Recargas rápidas:**
- Busca aumentos súbitos (delta>umbral)
- Verifica que la tendencia del combustible realmente aumento (comparando mediana móvil anterior/posterior), debido a que puede darse el caso de ser solo un delta alto mayor al umbral aislado y la tendencia no aumentase realmente.

**b) Recargas constantes:**
- Agrupa aumentos graduales consecutivos en una ventana de tiempo.
- Suma los aumentos graduales que pertenecen a una ventana de tiempo de maximo 3 horas
- Solo considera grupos con >400L de aumento total

> **Nota clave:** Todo el proceso considera la capacidad real de cada camión (ej: 3,200L + margen de seguridad) y el umbral mínimo de 190L por intervalo de 30 segundos establecido técnicamente.

## Correlacion de eventos de recarga de sensor con eventos registrados de la base de datos del surtidor

Este proceso compara dos fuentes de información para validar las recargas de combustible:  
1. **Datos del sensor** (eventos detectados del sensor de combustible)  
2. **Registros del surtidor** (eventos obtenidos de la base de datos del surtidor)  

#### Paso 1: Preparación de datos
- Se cargan ambos conjuntos de datos desde archivos CSV  
- Se filtran por rango de fechas en el rango disponible de ambos conjuntos de datos (Feb 2024 - Feb 2025)  
- Se clasifica cada evento por turno operativo:  
  - **D** (día: 6:00 - 17:59)  
  - **N** (noche: 18:00 - 5:59)
  Esta última decisión, que puede ser contraproducente ya que los turnos operativos de cambio se realizan a las 7 y no a las 6, se debe a que aporta mayor efectividad en la clasificación, debido a que los registros de los dieseleros se realizan manualmente y, con este pequeño cambio, se obtiene mayor precisión

#### Paso 2: Sistema de basado en el algoritmo fuel score

El algoritmo fuel score creado por el autor del reporte se basa en la idea de modelos del algoritmo `Credit score algorithm` que clasifica y toma desiciones basados en un puntaje obtenido en eventos previos, en este caso se tomando ambos eventos de recarga del sensor y los registrados y se les da puntaje basado en los siguientes criterios para otorgar 100% puntaje para relacionar un evento de recarga del sensor con un evento registrado de surtidor:

1. **Proximidad temporal (40%):** Eventos más cercanos en tiempo reciben mayor puntuación, el límite máximo para correlacionar eventos es de 12 horas.  
Sea:

- $t_r$ = tiempo del evento de recarga (RefillTimeStamp)  
- $t_s$ = tiempo del evento de suministro (SupplyTimeStamp)  
- $\Delta t = |t_r - t_s|$ = diferencia absoluta en horas entre ambos eventos  

donde:

- Si $\Delta t = 0$, entonces $\text{time\_score} = 1$ (máximo puntaje).  
- Si $\Delta t = 12$ horas, entonces $\text{time\_score} = 0$.  
- Para evitar valores negativos cuando $\Delta t > 12$, se puede limitar el puntaje mínimo a cero:

$$
\text{time\_score} = \max \left(0, 1 - \frac{\Delta t}{12} \right) \cdot 0.4
$$


- Ej: Recarga de sensor a las 14:30 y registro del surtidor a las 14:35 → 
$$
\Delta t = | t_r - t_s | = |14:30 - 14:35| = 5 \text{ minutos} = \frac{5}{60} = 0.0833 \text{ horas}
$$

$$
\text{time\_score} = \max \left(0, 1 - \frac{0.0833}{12} \right) \cdot 0.4 = (1 - 0.00694) \cdot 0.4 = 0.397224
$$

$$
\boxed{0.993}
$$

2. **Coincidencia de litros (35%):**  Compara diferencia entre litros detectados por el sensor y del surtidor para comparar la mayor similitud posible:  
Sea:

- $\Delta_f$ = litros detectados por el sensor (`delta_fuel`)  
- $F_L$ = litros registrados en surtidor (`FuelLevelLiters`)  

$$
d_{\text{litros}} = \frac{|\Delta_f - F_L|}{\max(\Delta_f, F_L)}
$$

$$
\text{liters\_score} = (1 - d_{\text{litros}}) \cdot 0.35
$$



3. **Confianza del origen (15%):** Definimos una función discreta que asigna puntajes según el origen:
- Registros de surtidores oficiales (`SURTIDOR-TRUCKSHOP`) reciben mayor puntaje  
- Orígenes de dieseleros reciben menor puntuación  

$$
\text{origin\_score} = 
\begin{cases}
1, & \text{si origen} = \text{SURTIDOR-TRUCKSHOP} \\
0.5, & \text{si otro origen} = \text{P068, SST} \\
\end{cases}
$$

4. **Coincidencia de turno (10%):** Empareja eventos ocurridos en mismo turno (D-D o N-N)  

Sea:

- $ T_r $ = turno del evento de recarga (D o N)  
- $ T_s $ = turno del evento de suministro (D o N)  

Definimos:

$$
\text{shift\_score} = 
\begin{cases}
1, & \text{si } T_r = T_s \\
0, & \text{si } T_r \neq T_s
\end{cases}
$$

**Puntaje total:**

$$
\text{total\_score} = \text{time\_score} + \text{liters\_score} + \text{origin\_score} + \text{shift\_score}
$$


#### Paso 3: Selección de mejores coincidencias
- Se ordenan todos los posibles emparejamientos por puntuación total  
- Se seleccionan las mejores coincidencias usando:  
  - **Algoritmo voraz:** Toma la mejor opción disponible sin repetir eventos  
  - **Evita duplicados:** Cada evento solo se empareja una vez  

#### Paso 4: Clasificación de resultados
Cada evento se categoriza como:

| Tipo | Descripción |
|------|-------------|
| **Both_Events** | Coincidencia exacta |
| **Refill_Only** | Detección sin registro |
| **Supply_Only** | Registro sin detección |

#### Paso 5: Análisis de discrepancias
Para eventos coincidentes se calculan columnas:
- **Diferencia temporal:** Segundos entre eventos de sensor y registro de surtidor
- **Diferencia de litros:** Variación de combustible entre eventos de sensor y registro de surtidor

## Analisis de resultados